# Phase B — Step 9: SpliceAI Analysis

**Why this step:** CADD scores poorly for splice site variants. PAPOLA (CADD=2.71) and TNRC18 (CADD=22.2) scored LOW/MODERATE in Step 8, but they are HIGH-impact splice variants. SpliceAI is purpose-built for splice disruption prediction and gives a more accurate picture for these 4 variants.

**4 HIGH-impact splice variants to score:**
- PAPOLA   chr14:97022168  (splice_acceptor_variant)
- SPICE1   chr3:113218285  C>G (splice_donor_variant)
- SPICE1   chr3:113218285  C>A (splice_donor_variant)
- TNRC18   chr7:5385191    (splice_donor_variant)

**Method:** SpliceAI via Illumina's free web-based prediction or local model. This notebook uses the SpliceAI Python package directly (no API needed).

**Output:** Delta scores (0-1) for acceptor gain/loss, donor gain/loss per variant. Score >0.5 = high confidence splice disruption.

**Runtime:** ~10-15 minutes (downloads reference genome data)

In [ ]:
!pip install spliceai pyfaidx -q
import warnings
warnings.filterwarnings('ignore')
print('SpliceAI installed')

In [ ]:
# SpliceAI requires reference genome FASTA and annotation
# Download hg19 reference for the specific genes only (much faster than full genome)
import os

os.makedirs('spliceai_data', exist_ok=True)
os.chdir('spliceai_data')

# Download GRCh37 reference genome (chr3, chr7, chr14 only — where our variants are)
print('Downloading reference chromosomes (3, 7, 14)...')
print('This is the most time-consuming step (~5-8 min)')

chroms_needed = ['3', '7', '14']
for chrom in chroms_needed:
    url = f'http://hgdownload.soe.ucsc.edu/goldenPath/hg19/chromosomes/chr{chrom}.fa.gz'
    print(f'Downloading chr{chrom}...')
    os.system(f'wget -q -O chr{chrom}.fa.gz "{url}"')
    os.system(f'gunzip -f chr{chrom}.fa.gz')
    if os.path.exists(f'chr{chrom}.fa'):
        size = os.path.getsize(f'chr{chrom}.fa') / 1e6
        print(f'  chr{chrom}.fa downloaded: {size:.1f} MB')
    else:
        print(f'  FAILED to download chr{chrom}')

os.chdir('..')

In [ ]:
# Combine chromosomes into single reference file for SpliceAI
import os

os.chdir('spliceai_data')
os.system('cat chr3.fa chr7.fa chr14.fa > reference.fa')
os.system('samtools faidx reference.fa 2>/dev/null || pip install pyfaidx -q')

from pyfaidx import Fasta
try:
    ref = Fasta('reference.fa')
    print(f'Reference loaded: {list(ref.keys())}')
except Exception as e:
    print(f'Error: {e}')
    print('Trying alternative indexing...')
    os.system('pip install pysam -q')

os.chdir('..')

In [ ]:
# Build VCF input file for the 4 splice variants
# SpliceAI requires VCF format input

vcf_content = '''##fileformat=VCFv4.2
##contig=<ID=3>
##contig=<ID=7>
##contig=<ID=14>
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
3\t113218285\tSPICE1_1\tC\tG\t.\t.\t.
3\t113218285\tSPICE1_2\tC\tA\t.\t.\t.
7\t5385191\tTNRC18\tA\tC\t.\t.\t.
14\t97022168\tPAPOLA\tAT\tATT\t.\t.\t.
'''

with open('splice_variants.vcf', 'w') as f:
    f.write(vcf_content)

print('VCF file created:')
print(vcf_content)

In [ ]:
# Run SpliceAI via command line tool
import os

# SpliceAI needs annotation file — download GRCh37 gene annotation
print('Downloading gene annotation...')
os.system('pip install spliceai -q')

# Run spliceai command
cmd = (
    'spliceai -I splice_variants.vcf '
    '-O splice_variants_scored.vcf '
    '-R spliceai_data/reference.fa '
    '-A grch37'
)
print(f'Running: {cmd}')
result = os.system(cmd)

if os.path.exists('splice_variants_scored.vcf'):
    print('SpliceAI completed successfully')
    with open('splice_variants_scored.vcf') as f:
        print(f.read())
else:
    print('SpliceAI run may have failed — check error above')
    print('If it failed, use the manual web-based fallback in next cell')

## FALLBACK — If SpliceAI command line fails

SpliceAI command-line setup in Colab is notoriously fragile (TensorFlow version conflicts, missing annotation files). **If Cell 6 failed**, use the official web-based SpliceAI lookup instead — it's the same underlying model and is what most published papers actually cite.

**Manual steps (5 minutes):**
1. Go to https://spliceailookup.broadinstitute.org/
2. Enter each variant in this format and note the scores:

| Gene | Input format | 
|------|-------------|
| SPICE1 (C>G) | `3-113218285-C-G` |
| SPICE1 (C>A) | `3-113218285-C-A` |
| TNRC18 | `7-5385191-A-C` |
| PAPOLA | `14-97022168-AT-ATT` |

3. Select genome version: **GRCh37/hg19**
4. Record the **DS_AG, DS_AL, DS_DG, DS_DL** scores (delta scores 0-1) for each
5. Paste your results into the next cell

In [ ]:
# Paste your SpliceAI Lookup results here
# DS_AG = Donor Acceptor Gain | DS_AL = Acceptor Loss
# DS_DG = Donor Gain | DS_DL = Donor Loss
# Max delta score across all 4 = overall splice disruption severity

import pandas as pd

# REPLACE these values with what you see on spliceailookup.broadinstitute.org
spliceai_results = pd.DataFrame({
    'gene'   : ['SPICE1', 'SPICE1', 'TNRC18', 'PAPOLA'],
    'variant': ['chr3:113218285 C>G', 'chr3:113218285 C>A',
                'chr7:5385191 A>C', 'chr14:97022168 AT>ATT'],
    'DS_AG'  : [0.00, 0.00, 0.00, 0.00],   # <- replace with real values
    'DS_AL'  : [0.00, 0.00, 0.00, 0.00],   # <- replace with real values
    'DS_DG'  : [0.00, 0.00, 0.00, 0.00],   # <- replace with real values
    'DS_DL'  : [0.91, 0.85, 1.00, 0.00],   # <- replace with real values
})

spliceai_results['max_delta'] = spliceai_results[['DS_AG','DS_AL','DS_DG','DS_DL']].max(axis=1)
spliceai_results['interpretation'] = spliceai_results['max_delta'].apply(
    lambda x: 'HIGH confidence splice disruption' if x >= 0.5
    else 'Moderate confidence' if x >= 0.2
    else 'Low confidence'
)

print('SPLICEAI RESULTS — 4 HIGH-IMPACT SPLICE VARIANTS')
print('='*80)
print(spliceai_results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs('figures_step9', exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(spliceai_results))
width = 0.2

ax.bar(x - 1.5*width, spliceai_results['DS_AG'], width, label='Acceptor Gain', color='#2E5FA3')
ax.bar(x - 0.5*width, spliceai_results['DS_AL'], width, label='Acceptor Loss', color='#5B9BD5')
ax.bar(x + 0.5*width, spliceai_results['DS_DG'], width, label='Donor Gain', color='#E67E22')
ax.bar(x + 1.5*width, spliceai_results['DS_DL'], width, label='Donor Loss', color='#C0392B')

ax.axhline(0.5, color='black', ls='--', lw=1.2, alpha=0.6, label='High confidence threshold')
ax.set_xticks(x)
ax.set_xticklabels(spliceai_results['gene'], fontsize=10)
ax.set_ylabel('SpliceAI Delta Score', fontsize=11)
ax.set_title('SpliceAI Delta Scores — 4 HIGH-Impact Splice Variants\n'
             'Scores ≥0.5 indicate high-confidence splice disruption',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, ncol=2)
ax.set_ylim(0, 1.1)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step9/spliceai_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('SpliceAI figure saved')

In [ ]:
from google.colab import drive
import shutil
from google.colab import files

drive.mount('/content/drive')

out_path = '/content/drive/My Drive/spliceai_results_step9.csv'
spliceai_results.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

shutil.make_archive('step9_outputs', 'zip', 'figures_step9')
files.download('step9_outputs.zip')
files.download(out_path)
print('Downloaded')
print()
print('NEXT: Step 12 — Cas-OFFinder verification for SPICE1')